# Exploratory Data Analysis (EDA)

**Input  :** `data/silver/` — cleaned files
**Output :** `reports/eda_summary.json` — key findings 
**Charts :** `reports/figures/` — all plots saved as PNG

---

## Sections

| Section | Question answered |
|---------|------------------|
| A | Dataset overview — shapes, memory, unique IDs |
| B | Volume distribution — histogram, box plot, monthly trends |
| C | Outlet analysis — top/bottom outlets, CV, max/mean ratio |
| D | Outlet master — type/size breakdown, cooler vs volume |
| E | Seasonality — do the labels match actual volume? |
| F | Holidays — which months are most disrupted? |
| G | Censoring summary — how much data is likely constrained? |

## Rule
> Every chart answers a specific question. No decoration.

## Imports

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

%matplotlib inline

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.eda_utils import (
    dataset_overview,
    plot_volume_distribution,
    plot_volume_by_month,
    compute_outlet_stats,
    plot_outlet_cv_distribution,
    plot_top_bottom_outlets,
    plot_outlet_categorical_breakdown,
    plot_cooler_vs_volume,
    plot_seasonality_index_distribution,
    plot_volume_by_seasonality,
    plot_holidays_per_month,
    censoring_signal_summary,
)

print("Imports OK")
print(f"Pandas     : {pd.__version__}")
print(f"Matplotlib : {plt.matplotlib.__version__}")

## Configuration

In [ ]:
from src.configs.config import config
from src.utils.io import read_parquet

os.makedirs(config.FIGURES_DIR, exist_ok=True)
os.makedirs(config.REPORTS_DIR, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

eda_findings = {
    "phase": "EDA",
    "run_timestamp": datetime.now().isoformat(),
    "findings": {},
}


def save_fig(fig, name):
    path = os.path.join(config.FIGURES_DIR, f"{name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  Saved: {path}")
    return path


print(f"Config loaded. Figures -> {config.FIGURES_DIR}")

## Load Silver Files

In [ ]:
silver_path = config.SILVER_PATH

tx = read_parquet(silver_path, "transactions_history_final")
out = read_parquet(silver_path, "outlet_master")
dist = read_parquet(silver_path, "distributor_seasonality_details")
hol = read_parquet(silver_path, "holiday_list")

hol["Date"] = pd.to_datetime(hol["Date"], errors="coerce")

print("Silver files loaded:")
print(f"  transactions            : {tx.shape[0]:>8,} rows x {tx.shape[1]} cols")
print(f"  outlet_master           : {out.shape[0]:>8,} rows x {out.shape[1]} cols")
print(f"  distributor_seasonality : {dist.shape[0]:>8,} rows x {dist.shape[1]} cols")
print(f"  holiday_list            : {hol.shape[0]:>8,} rows x {hol.shape[1]} cols")

---
## Section A — Dataset Overview

In [ ]:
frames = {
    "transactions": tx,
    "outlet_master": out,
    "distributor_seasonality": dist,
    "holiday_list": hol,
}

overview = dataset_overview(frames)
display(overview)

eda_findings["findings"]["dataset_overview"] = overview.to_dict(orient="records")

### A2 — Unique ID Counts

How many distinct outlets, distributors, and SKUs are we working with?

In [ ]:
id_counts = {
    "Unique Outlet_IDs in transactions": tx["Outlet_ID"].nunique(),
    "Unique Outlet_IDs in outlet_master": out["Outlet_ID"].nunique(),
    "Unique Distributor_IDs": tx["Distributor_ID"].nunique(),
    "Unique SKU_IDs": tx["SKU_ID"].nunique(),
    "Year range": f"{tx['Year'].min()} to {tx['Year'].max()}",
    "Months covered": tx["Month"].nunique(),
    "Total holidays in dataset": len(hol),
}

for k, v in id_counts.items():
    print(f"  {k:<45} {str(v):>10}")

eda_findings["findings"]["id_counts"] = {k: str(v) for k, v in id_counts.items()}

---
## Section B — Volume Distribution

**Question:** What does the shape of sales volume look like?
Is it normally distributed, skewed, or bimodal? Are there extreme outliers?

In [ ]:
nonzero_vol = tx[tx["Volume_Liters"] > 0]["Volume_Liters"]

vol_stats = {
    "total_rows": len(tx),
    "zero_volume_rows": int((tx["Volume_Liters"] == 0).sum()),
    "nonzero_rows": int((tx["Volume_Liters"] > 0).sum()),
    "mean": round(nonzero_vol.mean(), 2),
    "median": round(nonzero_vol.median(), 2),
    "std": round(nonzero_vol.std(), 2),
    "min": round(nonzero_vol.min(), 2),
    "max": round(nonzero_vol.max(), 2),
    "p95": round(nonzero_vol.quantile(0.95), 2),
    "p99": round(nonzero_vol.quantile(0.99), 2),
}

print("Volume_Liters Summary (non-zero rows only)\n")
for k, v in vol_stats.items():
    print(f"  {k:<25} {str(v):>12}")

zero_pct = 100 * vol_stats["zero_volume_rows"] / vol_stats["total_rows"]
print(
    f"\nZero-volume rows: {vol_stats['zero_volume_rows']:,} ({zero_pct:.1f}% of all transactions)"
)
print("These are potential censoring events.")

eda_findings["findings"]["volume_stats"] = vol_stats

In [ ]:
fig = plot_volume_distribution(tx)
save_fig(fig, "B1_volume_distribution")
plt.show()

### B2 — Monthly Volume Trend

**Question:** Is there a seasonal pattern? Does January tend to be high or low?

In [ ]:
fig = plot_volume_by_month(tx)
save_fig(fig, "B2_volume_by_month")
plt.show()

jan_avg = tx[tx["Month"] == 1]["Volume_Liters"].mean()
all_monthly_avg = (
    tx.groupby("Month")["Volume_Liters"].mean().sort_values(ascending=False)
)
jan_rank = all_monthly_avg.index.tolist().index(1) + 1

print(f"January average volume : {jan_avg:,.1f} L")
print(f"January rank by volume : {jan_rank} out of 12 months")
eda_findings["findings"]["january_volume_rank"] = jan_rank

---
## Section C — Outlet-Level Analysis

**Question:** Which outlets dominate volume? Are some outlets suspiciously flat (censored)?

In [ ]:
outlet_stats = compute_outlet_stats(tx)

print(f"Outlet stats computed for {len(outlet_stats):,} outlets")
print("\nTop 5 rows:")
display(outlet_stats.head())
print("\nSummary statistics:")
display(outlet_stats.describe().round(2))

In [ ]:
fig = plot_top_bottom_outlets(outlet_stats, n=20)
save_fig(fig, "C1_top_bottom_outlets")
plt.show()

### C2 — Coefficient of Variation (CV)

**Question:** How many outlets have suspiciously flat sales?

CV = std / mean. CV near 0 means near-identical sales every month — a delivery cap signal.
CV > 0.5 means highly variable — possibly intermittent supply.

In [ ]:
fig = plot_outlet_cv_distribution(outlet_stats)
save_fig(fig, "C2_outlet_cv_distribution")
plt.show()

flat_n = (outlet_stats["cv"] < 0.05).sum()
normal_n = ((outlet_stats["cv"] >= 0.05) & (outlet_stats["cv"] <= 0.5)).sum()
high_n = (outlet_stats["cv"] > 0.5).sum()
total_out = len(outlet_stats)

print(f"Flat   (CV < 0.05)       : {flat_n:,}  ({100 * flat_n / total_out:.1f}%)")
print(f"Normal (0.05 <= CV <= 0.5): {normal_n:,}  ({100 * normal_n / total_out:.1f}%)")
print(f"High   (CV > 0.5)        : {high_n:,}  ({100 * high_n / total_out:.1f}%)")

eda_findings["findings"]["cv_breakdown"] = {
    "flat_outlets": int(flat_n),
    "normal_outlets": int(normal_n),
    "high_cv_outlets": int(high_n),
}

### C3 — Max/Mean Ratio

**Question:** Did low-average outlets ever sell much more?

max / mean > 3 means the low months were likely constrained.
The maximum is the best available proxy for true potential.

In [ ]:
high_ratio = outlet_stats[outlet_stats["ratio_max_to_mean"] > 3].shape[0]
print(
    f"Outlets with max/mean ratio > 3x : {high_ratio:,} ({100 * high_ratio / len(outlet_stats):.1f}%)"
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(
    outlet_stats["ratio_max_to_mean"].clip(upper=20),
    bins=40,
    color="#2563EB",
    alpha=0.8,
    edgecolor="white",
)
ax.axvline(3, color="#DC2626", linestyle="--", linewidth=1.5, label="3x threshold")
ax.set_xlabel("Max / Mean Volume Ratio (clipped at 20)")
ax.set_ylabel("Number of Outlets")
ax.set_title("Max/Mean Volume Ratio Distribution", fontsize=12, fontweight="bold")
ax.legend()
fig.tight_layout()
save_fig(fig, "C3_max_mean_ratio")
plt.show()

eda_findings["findings"]["high_ratio_outlets"] = int(high_ratio)

---
## Section D — Outlet Master Analysis

**Question 1:** What is the type/size breakdown of outlets?
**Question 2:** Do outlets with more coolers sell more?

In [ ]:
fig = plot_outlet_categorical_breakdown(out)
save_fig(fig, "D1_outlet_categorical_breakdown")
plt.show()

In [ ]:
fig = plot_cooler_vs_volume(tx, out)
save_fig(fig, "D2_cooler_vs_volume")
plt.show()

outlet_mean = tx.groupby("Outlet_ID")["Volume_Liters"].mean().reset_index()
outlet_mean.columns = ["Outlet_ID", "mean_volume"]
merged_corr = outlet_mean.merge(
    out[["Outlet_ID", "Cooler_Count"]], on="Outlet_ID", how="inner"
)
corr_val = merged_corr["Cooler_Count"].corr(merged_corr["mean_volume"])

print(f"Pearson correlation (Cooler_Count vs mean volume): {corr_val:.4f}")
eda_findings["findings"]["cooler_volume_correlation"] = round(float(corr_val), 4)

---
## Section E — Seasonality Analysis

**Question 1:** What is the distribution of the Seasonality_Index labels?
**Question 2:** Do these labels actually correspond to higher/lower volumes?

In [ ]:
fig = plot_seasonality_index_distribution(dist)
save_fig(fig, "E1_seasonality_distribution")
plt.show()

dist_counts = dist["Seasonality_Index"].value_counts().to_dict()
for k, v in dist_counts.items():
    print(f"  {k:<15} {v:>6,}")
eda_findings["findings"]["seasonality_value_counts"] = {
    str(k): int(v) for k, v in dist_counts.items()
}

In [ ]:
fig = plot_volume_by_seasonality(tx, dist)
save_fig(fig, "E2_volume_by_seasonality")
plt.show()

merged_seas = tx.merge(
    dist[["Distributor_ID", "Year", "Month", "Seasonality_Index"]],
    on=["Distributor_ID", "Year", "Month"],
    how="left",
)
seas_medians = (
    merged_seas[merged_seas["Volume_Liters"] > 0]
    .groupby("Seasonality_Index")["Volume_Liters"]
    .median()
    .sort_values(ascending=False)
)
print("Median Volume_Liters by Seasonality_Index:")
print(seas_medians.round(1).to_string())
eda_findings["findings"]["seasonality_median_volume"] = seas_medians.round(1).to_dict()

---
## Section F — Holiday Analysis

**Question:** Which months have the most holidays?
High-holiday months may have disrupted deliveries contributing to censored demand.

In [ ]:
fig = plot_holidays_per_month(hol)
save_fig(fig, "F1_holidays_per_month")
plt.show()

print("Holiday Type breakdown:")
display(hol["Holiday_Type"].value_counts().reset_index())

jan_holidays = hol[hol["Month"] == 1]
print(f"\nJanuary holidays: {len(jan_holidays)}")
if len(jan_holidays) > 0:
    display(jan_holidays[["Date", "Holiday_Name", "Holiday_Type"]])

eda_findings["findings"]["january_holidays"] = len(jan_holidays)

---
## Section G — Censoring Signal Summary

Consolidates all censoring signals found across EDA.
This table directly motivates our modeling approach in Phase 6.

In [ ]:
censoring_df = censoring_signal_summary(tx, outlet_stats)

print("=" * 70)
print("CENSORING SIGNAL SUMMARY")
print("=" * 70)
display(censoring_df)

eda_findings["findings"]["censoring_summary"] = censoring_df.to_dict(orient="records")

### G2 — Outlet Trend Analysis

**Question:** Are outlets growing or declining?
Growing outlets should receive higher potential estimates.

In [ ]:
growing = (outlet_stats["trend_slope"] > 0).sum()
declining = (outlet_stats["trend_slope"] <= 0).sum()

print(f"Growing outlets  : {growing:,}  ({100 * growing / len(outlet_stats):.1f}%)")
print(f"Declining outlets: {declining:,}  ({100 * declining / len(outlet_stats):.1f}%)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(
    outlet_stats["trend_slope"].clip(-50, 50),
    bins=50,
    color="#2563EB",
    alpha=0.8,
    edgecolor="white",
)
ax.axvline(0, color="#DC2626", linestyle="--", linewidth=2, label="Zero (no trend)")
ax.set_xlabel("Trend Slope (Volume Liters / month, clipped +/-50)")
ax.set_ylabel("Number of Outlets")
ax.set_title("Outlet Volume Trend Distribution", fontsize=12, fontweight="bold")
ax.legend()
fig.tight_layout()
save_fig(fig, "G2_trend_slope_distribution")
plt.show()

eda_findings["findings"]["trend_breakdown"] = {
    "growing_outlets": int(growing),
    "declining_outlets": int(declining),
}